In [75]:
import yfinance as yf
import pandas as pd
import numpy as np

In [76]:
ticker_symbol = "AAPL"


apple_stock = yf.Ticker(ticker_symbol)


historical_data = apple_stock.history(period="10y")


print(historical_data)

                                 Open        High         Low       Close  \
Date                                                                        
2016-11-09 00:00:00-05:00   25.256985   25.587984   24.836343   25.486845   
2016-11-10 00:00:00-05:00   25.535117   25.535117   24.326056   24.776581   
2016-11-11 00:00:00-05:00   24.622575   25.024830   24.491555   24.923691   
2016-11-14 00:00:00-05:00   24.758201   24.781186   23.923810   24.298481   
2016-11-15 00:00:00-05:00   24.496157   24.751301   24.401915   24.620281   
...                               ...         ...         ...         ...   
2026-11-03 00:00:00-05:00  325.579987  334.500000  325.329987  331.899994   
2026-11-04 00:00:00-05:00  341.570007  354.500000  340.989990  349.510010   
2026-11-05 00:00:00-05:00  353.890015  355.380005  342.059998  344.170013   
2026-11-06 00:00:00-05:00  338.730011  345.790009  338.299988  342.369995   
2026-11-09 00:00:00-05:00  344.934998  350.334991  344.779999  349.339996   

In [77]:
historical_data['Date'] = historical_data.index

historical_data['Date']  = pd.to_datetime(historical_data['Date']).dt.date
historical_data.index = pd.to_datetime(historical_data['Date']).dt.date


In [78]:
historical_data = historical_data.drop(columns=['Dividends', 'Stock Splits', 'Date'])

In [85]:

rng = np.random.default_rng(10)         
rules = []

for i in range(500):
    kind = rng.choice(['random moving-average pairs', 'random RSI thresholds', 'random band widths'])
    if kind == 'random moving-average pairs':
        fast = int(rng.integers(5, 60))
        rules.append({'kind': 'random moving-average pairs', 'fast': fast, 'slow': int(rng.integers(fast + 10, 250))})
    elif kind == 'random RSI thresholds':
        rules.append({'kind': 'random RSI thresholds', 'n': int(rng.integers(5, 30)),
                      'buy': int(rng.integers(15, 40)), 'exit': int(rng.integers(45, 75))})
    else:
        rules.append({'kind': 'random band widths', 'n': int(rng.integers(10, 60)),
                      'k': round(float(rng.uniform(1.0, 3.0)), 2),
                      'mode': str(rng.choice(['reverandom RSI thresholdson', 'breakout']))})

rules = pd.DataFrame(rules)
rules['kind'].value_counts()

kind
random RSI thresholds          176
random band widths             163
random moving-average pairs    161
Name: count, dtype: int64

In [80]:

close = historical_data['Close']
COST = 0.0005  


def rsi(p, n):
    d = p.diff()
    return 100 - 100 / (1 + d.clip(lower=0).rolling(n).mean() / (-d).clip(lower=0).rolling(n).mean())

def sharpe(r):
    sd = r.std()
    s = r.mean() / sd * np.sqrt(252)
    return s.where(sd > 0, 0.0) if isinstance(s, pd.Series) else (s if sd > 0 else 0.0)

def position(rule, p):
    if rule['kind'] == 'random moving-average pairs':
        slow = p.rolling(int(rule['slow'])).mean()
        pos = (p.rolling(int(rule['fast'])).mean() > slow).astype(float)
        return pos.where(slow.notna(), 0.0)
    sig = pd.Series(np.nan, index=p.index)
    if rule['kind'] == 'random RSI thresholds':
        r = rsi(p, int(rule['n']))
        sig[r < rule['buy']] = 1.0
        sig[r > rule['exit']] = 0.0
    else:
        mid, sd = p.rolling(int(rule['n'])).mean(), p.rolling(int(rule['n'])).std()
        lower, upper = mid - rule['k'] * sd, mid + rule['k'] * sd
        if rule['mode'] == 'reversion':
            sig[p < lower] = 1.0; sig[p > upper] = 0.0
        else:
            sig[p > upper] = 1.0; sig[p < lower] = 0.0
    return sig.ffill().fillna(0.0)

def search(p):
    pos = pd.concat({i: position(row, p) for i, row in rules.iterrows()}, axis=1)
    r = p.pct_change().fillna(0.0)
    held = pos.shift(1).fillna(0.0)
    net = held.mul(r, axis=0) - held.diff().abs().fillna(held.abs()) * COST
    split = p.index[len(p) // 2]
    is_sh = sharpe(net[net.index < split])
    best = is_sh.idxmax()
    oos_sh = sharpe(net[net.index >= split])
    return {'best_rule': rules.loc[best, ['kind','fast','slow','n','buy','exit','k','mode']].dropna().to_dict(),
            'best_is_sharpe': is_sh[best],
            'median_is_sharpe': is_sh.median(),
            'out_of_sample_sharpe': sharpe(net[net.index >= split][best]),
            'share_is_sharpe_above_0.5': (is_sh > 0.5).mean(),
            'median_oos_sharpe': oos_sh.median(),
            'best_oos_rank_pct': (oos_sh < oos_sh[best]).mean()    
            
            }


     

In [81]:
sigma = close.pct_change().std()
rw = pd.Series(close.iloc[0] * np.exp(np.cumsum(rng.normal(-0.5 * sigma**2, sigma, len(close)))),
               index=close.index)

print('Real data:  ', search(close))
print('Random walk:', search(rw))

Real data:   {'best_rule': {'kind': 'random band widths', 'n': 11.0, 'k': 1.19, 'mode': 'breakout'}, 'best_is_sharpe': np.float64(1.9227061931273282), 'median_is_sharpe': np.float64(0.860247106900023), 'out_of_sample_sharpe': np.float64(0.7971358514103618), 'share_is_sharpe_above_0.5': np.float64(0.732), 'median_oos_sharpe': np.float64(0.5230890269876151), 'best_oos_rank_pct': np.float64(0.916)}
Random walk: {'best_rule': {'kind': 'random RSI thresholds', 'n': 29.0, 'buy': 16.0, 'exit': 59.0}, 'best_is_sharpe': np.float64(0.7805885301921157), 'median_is_sharpe': np.float64(-0.1708422337931551), 'out_of_sample_sharpe': 0.0, 'share_is_sharpe_above_0.5': np.float64(0.004), 'median_oos_sharpe': np.float64(-0.3192074776352278), 'best_oos_rank_pct': np.float64(0.86)}


C:\Users\Марк\AppData\Local\Temp\ipykernel_19260\4054083821.py:11: RuntimeWarning: invalid value encountered in scalar divide
  s = r.mean() / sd * np.sqrt(252)


In [82]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

results = pd.DataFrame({'Real data': search(close), 'Random walk': search(rw)})
results.loc['best_rule'] = results.loc['best_rule'].apply(
    lambda d: ', '.join(f'{k}={v:g}' if isinstance(v, float) else f'{k}={v}' for k, v in d.items()))
results = results.rename(index={
    'best_rule': 'Best rule (chosen in-sample)',
    'best_is_sharpe': 'Best in-sample Sharpe',
    'median_is_sharpe': 'Median in-sample Sharpe (all 500 rules)',
    'out_of_sample_sharpe': 'Best rule: out-of-sample Sharpe',
    'median_oos_sharpe': 'Median out-of-sample Sharpe (all 500 rules)',
    'share_is_sharpe_above_0.5': 'Share of rules with in-sample Sharpe > 0.5',
    'best_oos_rank_pct': 'Best rule: out-of-sample percentile',
})
results.index.name = 'Metric'

header_fill = PatternFill('solid', start_color='F4B183')   
first_col_fill = PatternFill('solid', start_color='FCE4D6') 
side = Side(style='thin', color='000000')
border = Border(left=side, right=side, top=side, bottom=side)

with pd.ExcelWriter('task4_results.xlsx', engine='openpyxl') as xw:
    results.to_excel(xw, sheet_name='Results', startrow=1)   
    ws = xw.sheets['Results']
    ws['A1'] = 'Data-mining demonstration: 500 random rules'
    ws['A1'].font = Font(name='Arial', size=12, bold=True)

    for row in ws.iter_rows(min_row=2, max_row=len(results) + 2):
        for c in row:
            c.border = border
            c.font = Font(name='Arial', size=10)
            if c.row == 2:                                    
                c.fill, c.font = header_fill, Font(name='Arial', size=10, bold=True)
                c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            elif c.column == 1:                              
                c.fill = first_col_fill
                c.alignment = Alignment(horizontal='left', vertical='center')
            else:                                           
                label = ws.cell(c.row, 1).value
                c.alignment = Alignment(horizontal='right', vertical='center')
                if c.row > 3:
                    c.number_format = '0.00%' if ('Share' in label or 'percentile' in label) else '0.00'
    ws.row_dimensions[2].height = 30
    ws.column_dimensions['A'].width = 45
    ws.column_dimensions['B'].width = 32
    ws.column_dimensions['C'].width = 32

C:\Users\Марк\AppData\Local\Temp\ipykernel_19260\4054083821.py:11: RuntimeWarning: invalid value encountered in scalar divide
  s = r.mean() / sd * np.sqrt(252)
